In [4]:
#!/usr/bin/env python
# coding: utf-8
"""
USDZAR FX Volatility Dashboard - Clean Version
Only uses required libraries for BQNT Terminal compatibility
"""

# =============================================================================
# REQUIRED IMPORTS ONLY
# =============================================================================
from functools import partial
import numpy as np
import pandas as pd
import ipywidgets as widgets
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import bql
from datetime import timedelta
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================

# Date setup
timestamp = pd.Timestamp('now')
today = pd.to_datetime(timestamp).strftime('%Y-%m-%d')
yesterday = pd.to_datetime(timestamp - pd.Timedelta(1, "d")).strftime('%Y-%m-%d')

# USDZAR FX Volatility Tickers
universe = [
    "USDZARV1M BGN Curncy", "USDZARV3M BGN Curncy", "USDZARV1Y BGN Curncy", "USDZARV1W BGN Curncy",  
    "USDZARV6M BGN Curncy", "USDZARV2W BGN Curncy", "USDZARV2M BGN Curncy", "USDZARV9M BGN Curncy",
    "USDZARV3W BGN Curncy", "USDZARV18M BGN Curncy", "USDZARV4M BGN Curncy", "USDZARVON BGN Curncy",
    "USDZARV2Y BGN Curncy", "USDZAR10B3M BGN Curncy", "USDZAR10B1M BGN Curncy", "USDZAR10B1W BGN Curncy",
    "USDZAR10B6M BGN Curncy", "USDZAR10B1Y BGN Curncy", "USDZAR10B2M BGN Curncy", "USDZAR10B2W BGN Curncy",
    "USDZAR10B3W BGN Curncy", "USDZAR10B4M BGN Curncy", "USDZAR10B9M BGN Curncy", "USDZAR10B2Y BGN Curncy",
    "USDZAR10B18M BGN Curncy", "USDZAR25B1M BGN Curncy", "USDZAR25B1W BGN Curncy", "USDZAR25B1Y BGN Curncy",
    "USDZAR25B2M BGN Curncy", "USDZAR25B2W BGN Curncy", "USDZAR25B3M BGN Curncy", "USDZAR25B3W BGN Curncy",
    "USDZAR25B4M BGN Curncy", "USDZAR25B6M BGN Curncy", "USDZAR25B9M BGN Curncy", "USDZAR25BON BGN Curncy",
    "USDZAR15B18M BGN Curncy", "USDZAR15B1M BGN Curncy", "USDZAR15B1W BGN Curncy", "USDZAR15B1Y BGN Curncy",
    "USDZAR15B2M BGN Curncy", "USDZAR15B2W BGN Curncy", "USDZAR15B2Y BGN Curncy", "USDZAR15B3M BGN Curncy",
    "USDZAR15B3W BGN Curncy", "USDZAR15B6M BGN Curncy", "USDZAR15B9M BGN Curncy", "USDZAR35B18M BGN Curncy",
    "USDZAR35B1M BGN Curncy", "USDZAR35B1W BGN Curncy", "USDZAR35B1Y BGN Curncy", "USDZAR35B2M BGN Curncy",
    "USDZAR35B2W BGN Curncy", "USDZAR35B2Y BGN Curncy", "USDZAR35B3M BGN Curncy", "USDZAR35B3W BGN Curncy",
    "USDZAR35B4M BGN Curncy", "USDZAR35B6M BGN Curncy", "USDZAR35B9M BGN Curncy", 
    "USDZAR25R3M BGN Curncy", "USDZAR25R1M BGN Curncy", "USDZAR10R1W BGN Curncy", "USDZAR25R6M BGN Curncy",
    "USDZAR10R2W BGN Curncy", "USDZAR10R3M BGN Curncy", "USDZAR10R1M BGN Curncy", "USDZAR10R2M BGN Curncy",
    "USDZAR10R3W BGN Curncy", "USDZAR10R4M BGN Curncy", "USDZAR25R1W BGN Curncy", "USDZAR25R1Y BGN Curncy",
    "USDZAR25R2W BGN Curncy", "USDZAR25R3W BGN Curncy", "USDZAR10R1Y BGN Curncy", "USDZAR10R6M BGN Curncy",
    "USDZAR10R9M BGN Curncy", "USDZAR25R2M BGN Curncy", "USDZAR25R9M BGN Curncy", "USDZAR25R4M BGN Curncy",
    "USDZAR25RON BGN Curncy", "USDZAR10R18M BGN Curncy", "USDZAR10R2Y BGN Curncy", "USDZAR15R18M BGN Curncy",
    "USDZAR15R1M BGN Curncy", "USDZAR15R1W BGN Curncy", "USDZAR15R1Y BGN Curncy", "USDZAR15R2M BGN Curncy",
    "USDZAR15R2W BGN Curncy", "USDZAR15R2Y BGN Curncy", "USDZAR15R3M BGN Curncy", "USDZAR15R3W BGN Curncy",
    "USDZAR15R6M BGN Curncy", "USDZAR15R9M BGN Curncy", "USDZAR25R18M BGN Curncy", "USDZAR25R2Y BGN Curncy",
    "USDZAR35R18M BGN Curncy", "USDZAR35R1M BGN Curncy", "USDZAR35R1W BGN Curncy", "USDZAR35R1Y BGN Curncy", 
    "USDZAR35R2M BGN Curncy", "USDZAR35R2W BGN Curncy", "USDZAR35R2Y BGN Curncy", "USDZAR35R3M BGN Curncy",
    "USDZAR35R3W BGN Curncy", "USDZAR35R4M BGN Curncy", "USDZAR35R6M BGN Curncy", "USDZAR35R9M BGN Curncy"
]

# FX Tenors - Business Day Convention
fx_tenors = {
    'ON': {'name': 'Overnight', 'days': 1},
    '1W': {'name': '1 Week', 'days': 5},
    '2W': {'name': '2 Weeks', 'days': 10},
    '3W': {'name': '3 Weeks', 'days': 15},
    '1M': {'name': '1 Month', 'days': 21},
    '2M': {'name': '2 Months', 'days': 42},
    '3M': {'name': '3 Months', 'days': 63},
    '4M': {'name': '4 Months', 'days': 84},
    '6M': {'name': '6 Months', 'days': 126},
    '9M': {'name': '9 Months', 'days': 189},
    '1Y': {'name': '1 Year', 'days': 252},
    '18M': {'name': '18 Months', 'days': 378},
    '2Y': {'name': '2 Years', 'days': 504}
}

delta_levels = ['10', '15', '25', '35', '50']

# =============================================================================
# DATA PROCESSING FUNCTIONS
# =============================================================================

def create_vol_surface_rename_mapping():
    """Create mapping for renaming volatility surface columns"""
    rename_dict = {}
    for ticker in universe:
        if ticker.startswith('USDZARV') and 'BGN Curncy' in ticker:
            tenor = ticker.replace('USDZARV', '').replace(' BGN Curncy', '')
            rename_dict[ticker] = f"USDZAR_{tenor}_ATM_IV"
        elif not ticker.startswith('USDZARV') and 'BGN Curncy' in ticker:
            parts = ticker.replace('USDZAR', '').replace(' BGN Curncy', '')
            if 'B' in parts:
                b_index = parts.find('B')
                delta = parts[:b_index]
                tenor = parts[b_index+1:]
                if delta and tenor:
                    rename_dict[ticker] = f"USDZAR_{tenor}_{delta}D_BF"
            elif 'R' in parts:
                r_index = parts.find('R')
                delta = parts[:r_index]
                tenor = parts[r_index+1:]
                if delta and tenor:
                    rename_dict[ticker] = f"USDZAR_{tenor}_{delta}D_RR"
    return rename_dict


def calculate_fx_realized_vol_business_days(df, fx_spot_column='ZAR BGN Curncy'):
    """Calculate realized volatility using business day convention"""
    if fx_spot_column not in df.columns:
        return df
    
    df_result = df.copy()
    log_returns = np.log(df[fx_spot_column] / df[fx_spot_column].shift(1))
    
    for tenor, info in fx_tenors.items():
        window = max(info['days'], 5)
        new_column_name = f'USDZAR_{tenor}_ATM_RV'
        rolling_std = log_returns.rolling(window=window).std()
        
        if info['days'] < 5:
            scaling_factor = np.sqrt(252 / info['days'])
        else:
            scaling_factor = np.sqrt(252)
            
        annualized_vol = round(rolling_std * scaling_factor * 100, 3)
        df_result[new_column_name] = annualized_vol
    
    return df_result


def process_fx_vol_surface_data():
    """Complete data processing pipeline for FX volatility surface"""
    print("🚀 Starting FX Volatility Surface data processing...")
    
    full_universe = ['ZAR BGN Curncy'] + universe
    print(f"   Processing {len(full_universe)} tickers")
    
    # BQL Setup
    bq = bql.Service()
    date_range = bq.func.range('2012-10-01', yesterday)
    px_last = {'price': bq.data.px_last(dates=date_range, fill='prev')}
    
    print("   Executing BQL request...")
    request = bql.Request(full_universe, px_last)
    response = bq.execute(request)
    
    df = pd.concat([item.df() for item in response])
    df.drop(columns=['CURRENCY'], inplace=True)
    df = df.reset_index().pivot(index='DATE', columns='ID', values='price')
    df.reset_index(inplace=True)
    df.columns.name = None
    df['DATE'] = pd.to_datetime(df['DATE'])
    df.set_index('DATE', inplace=True)
    df = df[df.index.weekday < 5]
    
    print(f"   Data loaded: {df.shape[0]} dates, {df.shape[1]} columns")
    
    if df.shape[0] == 0:
        print("❌ No data rows returned")
        return None
    
    # Rename columns
    fx_rename_mapping = create_vol_surface_rename_mapping()
    existing_fx_mapping = {old_name: new_name 
                          for old_name, new_name in fx_rename_mapping.items() 
                          if old_name in df.columns}
    df.rename(columns=existing_fx_mapping, inplace=True)
    
    # Calculate realized volatilities
    df_final = calculate_fx_realized_vol_business_days(df, 'ZAR BGN Curncy')
    
    print(f"✅ Processing complete: {df_final.shape}")
    return df_final


# =============================================================================
# DASHBOARD CLASS
# =============================================================================

class FXVolSurfaceDashboard:
    def __init__(self, df):
        self.df = df
        self.selected_cell = None
        self.setup_widgets()
        self.setup_layout()
    
    def setup_widgets(self):
        """Setup interactive widgets"""
        self.chart_type = widgets.Dropdown(
            options=[
                ('Volatility Surface Heatmap', 'vol_surface_heatmap'),
                ('Volatility Heatmap (ATM IV vs RV)', 'vol_heatmap'),
                ('Risk Reversal Heatmap', 'rr_heatmap'),
                ('Butterfly Heatmap', 'bf_heatmap'),
                ('ATM Vol Term Structure (IV vs RV)', 'atm_term_structure'),
                ('Risk Reversal Term Structure', 'rr_term_structure'),
                ('Butterfly Term Structure', 'bf_term_structure'),
                ('Volatility Smile by Tenor', 'vol_smile'),
                ('Smile Term Structure', 'smile_term_structure')
            ],
            value='vol_heatmap',
            description='Chart Type:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='350px')
        )
        
        self.delta_level = widgets.Dropdown(
            options=[('10 Delta', '10'), ('15 Delta', '15'), ('25 Delta', '25'), ('35 Delta', '35'), ('50 Delta (ATM)', '50')],
            value='25',
            description='Delta Level:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )
        
        self.tenor_select = widgets.Dropdown(
            options=[(f'{tenor} ({info["name"]})', tenor) for tenor, info in fx_tenors.items()],
            value='1M',
            description='Tenor:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='250px')
        )
        
        self.lookback_period = widgets.Dropdown(
            options=[
                ('Current', 0),
                ('3 Months Ago', 3),
                ('6 Months Ago', 6),
                ('12 Months Ago', 12),
                ('24 Months Ago', 24),
                ('60 Months Ago', 60),
                ('120 Months Ago', 120),
                ('Full Period', 'full')
            ],
            value=0,
            description='Time Period:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )
        
        self.update_button = widgets.Button(
            description='Refresh',
            button_style='success',
            layout=widgets.Layout(width='100px')
        )
        
        # Loading spinner
        self.spinner = widgets.HTML(
            '''<i class="fa fa-spinner fa-spin" style="font-size: 18px"></i>''',
            layout=widgets.Layout(visibility='hidden', margin='12px 0 0 10px')
        )
        
        # Exception message box
        self.exception_box = widgets.HBox(layout=widgets.Layout(margin='6px 0 0 10px'))
        
        # Figure containers - use VBox to hold FigureWidgets
        self.main_fig_box = widgets.VBox()
        self.time_series_fig_box = widgets.VBox()
        
        # Event handlers
        self.chart_type.observe(self.on_widget_change, names='value')
        self.delta_level.observe(self.on_widget_change, names='value')
        self.tenor_select.observe(self.on_widget_change, names='value')
        self.lookback_period.observe(self.on_widget_change, names='value')
        self.update_button.on_click(self.on_button_click)
    
    def setup_layout(self):
        """Setup dashboard layout"""
        title = widgets.HTML(
            value="<h2 style='text-align: center; color: #2E86AB;'>USDZAR FX Volatility Surface Dashboard</h2>"
        )
        
        description = widgets.HTML(
            value="""
            <div style='text-align: center; margin-bottom: 20px; color: #666;'>
                <p><strong>ATM Volatilities • Risk Reversals • Butterflies • call_vol = atm + 0.5 * rr + bf  •  put_vol = atm - 0.5 * rr + bf</strong></p>
                <p>Business day convention: 1W=5d, 1M=21d, 1Y=252d</p>
            </div>
            """
        )
        
        controls_box = widgets.HBox([
            self.chart_type,
            self.delta_level,
            self.tenor_select,
            self.lookback_period
        ], layout=widgets.Layout(
            justify_content='center',
            align_items='center',
            margin='10px 0px'
        ))
        
        button_box = widgets.HBox([
            self.update_button, 
            self.spinner, 
            self.exception_box
        ], layout=widgets.Layout(
            justify_content='center',
            margin='10px 0px'
        ))
        
        main_chart_box = widgets.VBox([
            widgets.HTML(value="<h3 style='text-align: center; margin: 10px 0;'>Main Chart</h3>"),
            self.main_fig_box
        ])
        
        time_series_box = widgets.VBox([
            widgets.HTML(value="<h3 style='text-align: center; margin: 10px 0;'>Time Series Analysis</h3>"),
            self.time_series_fig_box
        ])
        
        charts_box = widgets.VBox([main_chart_box, time_series_box])
        
        self.dashboard = widgets.VBox([
            title,
            description,
            controls_box,
            button_box,
            charts_box
        ])
        
        # Run on startup
        self.run()
    
    def on_widget_change(self, change):
        """Widget change handler"""
        self.run()
    
    def on_button_click(self, button):
        """Handle button clicks"""
        self.run()
    
    def run(self, *args):
        """Main run function to update all charts"""
        try:
            self.spinner.layout.visibility = 'visible'
            self.exception_box.children = []
            
            self.update_chart()
            
            # Show automatic time series based on chart type
            chart_type = self.chart_type.value
            if chart_type == 'vol_heatmap':
                self.show_time_series_for_tenor(self.tenor_select.value)
            elif chart_type in ['rr_heatmap', 'bf_heatmap']:
                selected_tenor = self.tenor_select.value
                selected_delta = self.delta_level.value
                if selected_delta != '50':
                    self.show_rr_bf_time_series(selected_tenor, selected_delta)
            elif chart_type == 'smile_term_structure':
                selected_tenor = self.tenor_select.value
                selected_delta = self.delta_level.value
                self.show_smile_time_series(selected_tenor, selected_delta)
            else:
                self.time_series_fig_box.children = []
                
        except Exception as e:
            self.exception_box.children = [
                widgets.HTML(f'<span style="color: red;">Error: {str(e)}</span>')
            ]
        finally:
            self.spinner.layout.visibility = 'hidden'
    
    def get_historical_date(self, lookback_months):
        """Get historical date with proper error handling"""
        if self.df.empty:
            return None
            
        if lookback_months == 0:
            return self.df.index[-1]
        elif lookback_months == 'full':
            return self.df.index[0]
        
        target_date = self.df.index[-1] - timedelta(days=lookback_months * 30)
        available_dates = self.df.index[self.df.index <= target_date]
        return available_dates[-1] if len(available_dates) > 0 else self.df.index[0]
    
    def calculate_percentile(self, series, current_value, lookback_days):
        """Calculate percentile of current value over lookback period"""
        if pd.isna(current_value):
            return np.nan
        
        end_date = series.index[-1]
        start_date = end_date - timedelta(days=lookback_days)
        lookback_data = series[series.index >= start_date].dropna()
        
        if len(lookback_data) == 0:
            return np.nan
        
        percentile = (lookback_data <= current_value).mean() * 100
        return percentile

    def create_vol_surface_heatmap(self, lookback_months):
        """Create volatility surface heatmap"""
        target_date = self.get_historical_date(lookback_months)
        if target_date is None:
            return self.create_empty_chart("No data available")
        
        strike_order = ['35DP', '25DP', '15DP', '10DP', 'ATM', '10DC', '15DC', '25DC', '35DC']
        strike_labels = ['35Δ Put', '25Δ Put', '15Δ Put', '10Δ Put', 'ATM', '10Δ Call', '15Δ Call', '25Δ Call', '35Δ Call']
        tenors = ['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
        
        if lookback_months == 'full':
            lookback_days = len(self.df)
            show_percentiles = False
        else:
            lookback_days = lookback_months * 30
            show_percentiles = True
        
        z_matrix = []
        hover_matrix = []
        annotations = []
        
        for i, tenor in enumerate(tenors):
            row = []
            hover_row = []
            
            for j, strike in enumerate(strike_order):
                vol_val = np.nan
                percentile_val = np.nan
                
                if strike == 'ATM':
                    atm_col = f'USDZAR_{tenor}_ATM_IV'
                    if atm_col in self.df.columns:
                        atm_series = self.df[atm_col].loc[:target_date].dropna()
                        if len(atm_series) > 0:
                            vol_val = atm_series.iloc[-1]
                            if show_percentiles:
                                percentile_val = self.calculate_percentile(atm_series, vol_val, lookback_days)
                else:
                    delta = strike[:2]
                    option_type = strike[2:]
                    
                    atm_col = f'USDZAR_{tenor}_ATM_IV'
                    rr_col = f'USDZAR_{tenor}_{delta}D_RR'
                    bf_col = f'USDZAR_{tenor}_{delta}D_BF'
                    
                    if (atm_col in self.df.columns and rr_col in self.df.columns and bf_col in self.df.columns):
                        atm_data = self.df[atm_col].loc[:target_date].dropna()
                        rr_data = self.df[rr_col].loc[:target_date].dropna()
                        bf_data = self.df[bf_col].loc[:target_date].dropna()
                        
                        if len(atm_data) > 0 and len(rr_data) > 0 and len(bf_data) > 0:
                            atm_vol = atm_data.iloc[-1]
                            rr_vol = rr_data.iloc[-1]
                            bf_vol = bf_data.iloc[-1]
                            
                            if option_type == 'DC':
                                vol_val = atm_vol + 0.5 * rr_vol + bf_vol
                            elif option_type == 'DP':
                                vol_val = atm_vol - 0.5 * rr_vol + bf_vol
                            
                            if show_percentiles:
                                percentile_val = self.calculate_percentile(atm_data, atm_vol, lookback_days)
                
                row.append(vol_val)
                
                if not pd.isna(vol_val):
                    hover_row.append(f"Tenor: {tenor}<br>Strike: {strike_labels[j]}<br>Vol: {vol_val:.1f}%")
                    
                    if show_percentiles and not pd.isna(percentile_val):
                        annotation_text = f"<b>{vol_val:.1f}</b><br>{percentile_val:.1f}"
                    else:
                        annotation_text = f"<b>{vol_val:.1f}</b>"
                    
                    annotations.append(
                        dict(x=j, y=i, text=annotation_text, showarrow=False,
                            font=dict(size=9, color='black'), align='center')
                    )
                else:
                    hover_row.append(f"Tenor: {tenor}<br>Strike: {strike_labels[j]}<br>No Data")
            
            z_matrix.append(row)
            hover_matrix.append(hover_row)
        
        fig = go.Figure(data=go.Heatmap(
            z=z_matrix,
            x=strike_labels,
            y=tenors,
            colorscale='RdYlBu_r',
            text=hover_matrix,
            hovertemplate='%{text}<extra></extra>',
            colorbar=dict(title="Implied Volatility (%)", titleside="right")
        ))
        
        title_suffix = "Full Period" if lookback_months == 'full' else f"{lookback_months} Month Lookback"
        
        fig.update_layout(
            annotations=annotations,
            title=f"USDZAR Volatility Surface - {title_suffix}",
            xaxis=dict(title="Strike (Delta)", side="bottom"),
            yaxis=dict(title="Option Tenor", side="left"),
            width=1200, height=600
        )
        
        if show_percentiles:
            fig.add_annotation(
                text="<b>IV</b><br>IV Pctl<br>IV = % points<br>Pctl = percentile rank",
                xref="paper", yref="paper", x=1.05, y=1.15,
                showarrow=False, font=dict(size=9), align="left",
                bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
            )
        
        return fig

    def create_volatility_heatmap(self, lookback_months):
        """Create ATM volatility heatmap"""
        tenors = ['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
        deltas = ['ATM']
        
        if lookback_months == 'full':
            lookback_days = len(self.df)
        else:
            lookback_days = lookback_months * 30
        
        current_iv = np.full((len(deltas), len(tenors)), np.nan)
        hover_text = np.full((len(deltas), len(tenors)), "", dtype=object)
        annotations = []
        
        for i, delta in enumerate(deltas):
            for j, tenor in enumerate(tenors):
                if tenor in fx_tenors:
                    iv_col = f'USDZAR_{tenor}_ATM_IV'
                    rv_col = f'USDZAR_{tenor}_ATM_RV'
                    
                    if iv_col in self.df.columns and rv_col in self.df.columns:
                        iv_series = self.df[iv_col].dropna()
                        rv_series = self.df[rv_col].dropna()
                        
                        if len(iv_series) > 0 and len(rv_series) > 0:
                            curr_iv = iv_series.iloc[-1]
                            curr_rv = rv_series.iloc[-1]
                            
                            current_iv[i, j] = curr_iv
                            
                            hover_text[i, j] = (
                                f"Tenor: {tenor}<br>"
                                f"Current IV: {curr_iv:.1f}%<br>"
                                f"Current RV: {curr_rv:.1f}%<br>"
                                f"<i>Click to view time series</i>"
                            )
                            
                            if lookback_months != 'full':
                                perc_iv = self.calculate_percentile(iv_series, curr_iv, lookback_days)
                                perc_rv = self.calculate_percentile(rv_series, curr_rv, lookback_days)
                                
                                annotation_text = (
                                    f"<b>{curr_iv:.1f}</b>  <b>{curr_rv:.1f}</b><br>"
                                    f"{perc_iv:.1f}  {perc_rv:.1f}"
                                )
                            else:
                                annotation_text = f"<b>{curr_iv:.1f}</b>  <b>{curr_rv:.1f}</b>"
                            
                            annotations.append(
                                dict(x=j, y=i, text=annotation_text, showarrow=False,
                                    font=dict(size=9, color='black'), align='center')
                            )
        
        fig = go.Figure(data=go.Heatmap(
            z=current_iv,
            x=tenors,
            y=deltas,
            colorscale='RdYlBu_r',
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
            colorbar=dict(title="Current IV (%)", titleside="right")
        ))
        
        title_suffix = "Full Period" if lookback_months == 'full' else f"{lookback_months} Month Lookback"
        
        fig.update_layout(
            annotations=annotations,
            title=f"USDZAR ATM Volatility Grid - {title_suffix}",
            xaxis=dict(title="Option Tenor", side="bottom"),
            yaxis=dict(title="Strike", side="left"),
            width=1200, height=300
        )
        
        if lookback_months != 'full':
            fig.add_annotation(
                text="<b>IV</b>  <b>RV</b><br>IV Pctl  RV Pctl<br>IV/RV = % points<br>Pctl = percentile rank",
                xref="paper", yref="paper", x=1.05, y=1.15,
                showarrow=False, font=dict(size=9), align="left",
                bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
            )
        
        return fig
    
    def create_rr_heatmap(self, lookback_months):
        """Create Risk Reversal heatmap"""
        tenors = ['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
        deltas = ['10D', '15D', '25D', '35D']
        
        if lookback_months == 'full':
            lookback_days = len(self.df)
        else:
            lookback_days = lookback_months * 30
        
        current_rr = np.full((len(deltas), len(tenors)), np.nan)
        hover_text = np.full((len(deltas), len(tenors)), "", dtype=object)
        annotations = []
        
        for i, delta in enumerate(deltas):
            delta_num = delta.replace('D', '')
            for j, tenor in enumerate(tenors):
                if tenor in fx_tenors:
                    rr_col = f'USDZAR_{tenor}_{delta_num}D_RR'
                    
                    if rr_col in self.df.columns:
                        rr_series = self.df[rr_col].dropna()
                        
                        if len(rr_series) > 0:
                            curr_rr = rr_series.iloc[-1]
                            current_rr[i, j] = curr_rr
                            
                            hover_text[i, j] = (
                                f"Tenor: {tenor}, Delta: {delta}<br>"
                                f"Risk Reversal: {curr_rr:.1f}vol pts<br>"
                                f"<i>Click to view time series</i>"
                            )
                            
                            if lookback_months != 'full':
                                perc_rr = self.calculate_percentile(rr_series, curr_rr, lookback_days)
                                annotation_text = f"<b>{curr_rr:.1f}</b><br>{perc_rr:.1f}"
                            else:
                                annotation_text = f"<b>{curr_rr:.1f}</b>"
                            
                            annotations.append(
                                dict(x=j, y=i, text=annotation_text, showarrow=False,
                                    font=dict(size=9, color='black'), align='center')
                            )
        
        fig = go.Figure(data=go.Heatmap(
            z=current_rr,
            x=tenors,
            y=deltas,
            colorscale='RdBu_r',
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
            colorbar=dict(title="Risk Reversal (vol pts)", titleside="right"),
            zmid=0
        ))
        
        title_suffix = "Full Period" if lookback_months == 'full' else f"{lookback_months} Month Lookback"
        
        fig.update_layout(
            annotations=annotations,
            title=f"USDZAR Risk Reversal Grid - {title_suffix}",
            xaxis=dict(title="Option Tenor", side="bottom"),
            yaxis=dict(title="Delta Level", side="left"),
            width=1200, height=400
        )
        
        return fig
    
    def create_bf_heatmap(self, lookback_months):
        """Create Butterfly heatmap"""
        tenors = ['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
        deltas = ['10D', '15D', '25D', '35D']
        
        if lookback_months == 'full':
            lookback_days = len(self.df)
        else:
            lookback_days = lookback_months * 30
        
        current_bf = np.full((len(deltas), len(tenors)), np.nan)
        hover_text = np.full((len(deltas), len(tenors)), "", dtype=object)
        annotations = []
        
        for i, delta in enumerate(deltas):
            delta_num = delta.replace('D', '')
            for j, tenor in enumerate(tenors):
                if tenor in fx_tenors:
                    bf_col = f'USDZAR_{tenor}_{delta_num}D_BF'
                    
                    if bf_col in self.df.columns:
                        bf_series = self.df[bf_col].dropna()
                        
                        if len(bf_series) > 0:
                            curr_bf = bf_series.iloc[-1]
                            current_bf[i, j] = curr_bf
                            
                            hover_text[i, j] = (
                                f"Tenor: {tenor}, Delta: {delta}<br>"
                                f"Butterfly: {curr_bf:.1f}vol pts<br>"
                                f"<i>Click to view time series</i>"
                            )
                            
                            if lookback_months != 'full':
                                perc_bf = self.calculate_percentile(bf_series, curr_bf, lookback_days)
                                annotation_text = f"<b>{curr_bf:.1f}</b><br>{perc_bf:.1f}"
                            else:
                                annotation_text = f"<b>{curr_bf:.1f}</b>"
                            
                            annotations.append(
                                dict(x=j, y=i, text=annotation_text, showarrow=False,
                                    font=dict(size=9, color='black'), align='center')
                            )
        
        fig = go.Figure(data=go.Heatmap(
            z=current_bf,
            x=tenors,
            y=deltas,
            colorscale='Viridis',
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
            colorbar=dict(title="Butterfly (vol pts)", titleside="right")
        ))
        
        title_suffix = "Full Period" if lookback_months == 'full' else f"{lookback_months} Month Lookback"
        
        fig.update_layout(
            annotations=annotations,
            title=f"USDZAR Butterfly Grid - {title_suffix}",
            xaxis=dict(title="Option Tenor", side="bottom"),
            yaxis=dict(title="Delta Level", side="left"),
            width=1200, height=400
        )
        
        return fig
    
    def create_atm_term_structure(self):
        """Create ATM volatility term structure"""
        target_date = self.get_historical_date(self.lookback_period.value)
        if target_date is None:
            return self.create_empty_chart("No data available")
    
        if self.lookback_period.value == 0:
            box_lookback_months = 3
        elif self.lookback_period.value == 'full':
            box_lookback_months = 'full'
        else:
            box_lookback_months = self.lookback_period.value
    
        if box_lookback_months == 'full':
            box_start_date = self.df.index[0]
        else:
            box_start_date = target_date - timedelta(days=box_lookback_months * 30)
        
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=(
                f'USDZAR ATM Volatility Term Structure - {target_date.strftime("%Y-%m-%d")}',
                f'IV & RV Distribution ({box_lookback_months} Month{"s" if box_lookback_months != 1 and box_lookback_months != "full" else ""} Lookback)'
            ),
            vertical_spacing=0.12,
            row_heights=[0.6, 0.4]
        )

        tenors = list(fx_tenors.keys())

        atm_vols = []
        atm_labels = []
        atm_days = []
        for tenor in tenors:
            col_name = f'USDZAR_{tenor}_ATM_IV'
            if col_name in self.df.columns:
                data = self.df[col_name].loc[:target_date].dropna()
                if len(data) > 0:
                    atm_vols.append(data.iloc[-1])
                    atm_labels.append(tenor)
                    atm_days.append(fx_tenors[tenor]['days'])
            
        if atm_vols:
            fig.add_trace(go.Scatter(
                x=atm_days,
                y=atm_vols,
                mode='lines+markers',
                name='50Δ (ATM) Implied Vol',
                line=dict(color='blue', width=3),
                marker=dict(size=8),
                text=atm_labels,
                hovertemplate='<b>%{text}</b><br>ATM IV: %{y:.1f}%<extra></extra>',
                showlegend=True
            ), row=1, col=1)

        rv_vols = []
        rv_labels = []
        rv_days = []
        for tenor in tenors:
            col_name = f'USDZAR_{tenor}_ATM_RV'
            if col_name in self.df.columns:
                data = self.df[col_name].loc[:target_date].dropna()
                if len(data) > 0:
                    rv_vols.append(data.iloc[-1])
                    rv_labels.append(tenor)
                    rv_days.append(fx_tenors[tenor]['days'])
            
        if rv_vols:
            fig.add_trace(go.Scatter(
                x=rv_days,
                y=rv_vols,
                mode='lines+markers',
                name='50Δ (ATM) Realized Vol',
                line=dict(color='red', width=3, dash='dash'),
                marker=dict(size=8),
                text=rv_labels,
                hovertemplate='<b>%{text}</b><br>ATM RV: %{y:.1f}%<extra></extra>',
                showlegend=True
            ), row=1, col=1)

        key_tenors_for_box = ['1W', '1M', '3M', '6M', '1Y']
        box_colors = ['lightblue', 'lightcoral']

        for i, vol_type in enumerate(['IV', 'RV']):
            for j, tenor in enumerate(key_tenors_for_box):
                col_suffix = 'ATM_IV' if vol_type == 'IV' else 'ATM_RV'
                col_name = f'USDZAR_{tenor}_{col_suffix}'
        
                if col_name in self.df.columns:
                    if box_lookback_months == 'full':
                        box_data = self.df[col_name].dropna()
                    else:
                        box_data = self.df[col_name].loc[box_start_date:target_date].dropna()
            
                    if len(box_data) > 5:
                        fig.add_trace(go.Box(
                            y=box_data.values,
                            name=f'{tenor} {vol_type}',
                            x=[f'{tenor}<br>{vol_type}'] * len(box_data),
                            marker_color=box_colors[i],
                            line_color='darkblue' if vol_type == 'IV' else 'darkred',
                            showlegend=False,
                            boxpoints='outliers'
                        ), row=2, col=1)

        fig.update_layout(
            height=800,
            width=1200,
            plot_bgcolor='white',
            paper_bgcolor='white',
            hovermode='closest'
        )

        fig.update_xaxes(
            title='Tenor (Business Days)', 
            type='log',
            tickmode='array',
            tickvals=[1, 5, 10, 15, 21, 42, 63, 84, 126, 189, 252, 378, 504],
            ticktext=['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y'],
            row=1, col=1
        )

        fig.update_xaxes(title='Tenor & Type', row=2, col=1)
        fig.update_yaxes(title='Volatility (%)', row=1, col=1)
        fig.update_yaxes(title='Volatility (%)', row=2, col=1)
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

        return fig
    
    def create_rr_term_structure(self):
        """Create Risk Reversal term structure"""
        target_date = self.get_historical_date(self.lookback_period.value)
        if target_date is None:
            return self.create_empty_chart("No data available")
            
        fig = go.Figure()
        
        tenors = list(fx_tenors.keys())
        colors = ['green', 'orange', 'purple', 'brown']
        rr_deltas = ['10', '15', '25', '35']
        
        for i, delta in enumerate(rr_deltas):
            rr_values = []
            rr_labels = []
            rr_days = []
            
            for tenor in tenors:
                col_name = f'USDZAR_{tenor}_{delta}D_RR'
                if col_name in self.df.columns:
                    data = self.df[col_name].loc[:target_date].dropna()
                    if len(data) > 0:
                        rr_values.append(data.iloc[-1])
                        rr_labels.append(tenor)
                        rr_days.append(fx_tenors[tenor]['days'])
            
            if rr_values:
                fig.add_trace(go.Scatter(
                    x=rr_days,
                    y=rr_values,
                    mode='lines+markers',
                    name=f'{delta}Δ Risk Reversal',
                    line=dict(color=colors[i], width=3),
                    marker=dict(size=8),
                    text=rr_labels,
                    hovertemplate=f'<b>%{{text}} {delta}Δ</b><br>RR: %{{y:.1f}}vol pts<extra></extra>'
                ))
        
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
        
        fig.update_layout(
            title=f'USDZAR Risk Reversal Term Structure - {target_date.strftime("%Y-%m-%d")}',
            xaxis=dict(
                title='Tenor (Business Days)', 
                type='log',
                tickmode='array',
                tickvals=[1, 5, 10, 15, 21, 42, 63, 84, 126, 189, 252, 378, 504],
                ticktext=['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
            ),
            yaxis=dict(title='Risk Reversal (vol points)'),
            width=1000, height=600,
            hovermode='closest',
            plot_bgcolor='white'
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        return fig
    
    def create_bf_term_structure(self):
        """Create Butterfly term structure"""
        target_date = self.get_historical_date(self.lookback_period.value)
        if target_date is None:
            return self.create_empty_chart("No data available")
            
        fig = go.Figure()
        
        tenors = list(fx_tenors.keys())
        colors = ['purple', 'orange', 'green', 'brown']
        bf_deltas = ['10', '15', '25', '35']
        
        for i, delta in enumerate(bf_deltas):
            bf_values = []
            bf_labels = []
            bf_days = []
            
            for tenor in tenors:
                col_name = f'USDZAR_{tenor}_{delta}D_BF'
                if col_name in self.df.columns:
                    data = self.df[col_name].loc[:target_date].dropna()
                    if len(data) > 0:
                        bf_values.append(data.iloc[-1])
                        bf_labels.append(tenor)
                        bf_days.append(fx_tenors[tenor]['days'])
            
            if bf_values:
                fig.add_trace(go.Scatter(
                    x=bf_days,
                    y=bf_values,
                    mode='lines+markers',
                    name=f'{delta}Δ Butterfly',
                    line=dict(color=colors[i], width=3),
                    marker=dict(size=8),
                    text=bf_labels,
                    hovertemplate=f'<b>%{{text}} {delta}Δ</b><br>BF: %{{y:.1f}}vol pts<extra></extra>'
                ))
        
        fig.update_layout(
            title=f'USDZAR Butterfly Term Structure - {target_date.strftime("%Y-%m-%d")}',
            xaxis=dict(
                title='Tenor (Business Days)', 
                type='log',
                tickmode='array',
                tickvals=[1, 5, 10, 15, 21, 42, 63, 84, 126, 189, 252, 378, 504],
                ticktext=['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
            ),
            yaxis=dict(title='Butterfly (vol points)'),
            width=1000, height=600,
            hovermode='closest',
            plot_bgcolor='white'
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        return fig
    
    def create_vol_smile_by_tenor(self):
        """Create volatility smile with correct delta ordering"""
        target_date = self.get_historical_date(self.lookback_period.value)
        if target_date is None:
            return self.create_empty_chart("No data available")
        
        fig = go.Figure()
        
        tenors = ['1W', '1M', '3M', '6M', '1Y']
        colors = ['blue', 'red', 'green', 'purple', 'orange']
        
        delta_order = ['10', '15', '25', '35']
        strike_positions = []
        strike_labels = []
        
        pos = 0
        for delta in delta_order:
            strike_positions.append(pos)
            strike_labels.append(f'{delta}Δ Put')
            pos += 1
        
        strike_positions.append(pos)
        strike_labels.append('ATM')
        atm_pos = pos
        pos += 1
        
        for delta in delta_order[::-1]:
            strike_positions.append(pos)
            strike_labels.append(f'{delta}Δ Call')
            pos += 1
        
        for i, tenor in enumerate(tenors):
            x_positions = []
            y_values = []
            hover_labels = []
            
            atm_col = f'USDZAR_{tenor}_ATM_IV'
            if atm_col in self.df.columns:
                atm_data = self.df[atm_col].loc[:target_date].dropna()
                if len(atm_data) > 0:
                    atm_vol = atm_data.iloc[-1]
                    x_positions.append(atm_pos)
                    y_values.append(atm_vol)
                    hover_labels.append('ATM')
            
            for j, delta in enumerate(delta_order):
                atm_col = f'USDZAR_{tenor}_ATM_IV'
                rr_col = f'USDZAR_{tenor}_{delta}D_RR'
                bf_col = f'USDZAR_{tenor}_{delta}D_BF'
                
                if all(col in self.df.columns for col in [atm_col, rr_col, bf_col]):
                    atm_data = self.df[atm_col].loc[:target_date].dropna()
                    rr_data = self.df[rr_col].loc[:target_date].dropna()
                    bf_data = self.df[bf_col].loc[:target_date].dropna()
                    
                    if len(atm_data) > 0 and len(rr_data) > 0 and len(bf_data) > 0:
                        atm_vol_calc = atm_data.iloc[-1]
                        rr_vol = rr_data.iloc[-1]
                        bf_vol = bf_data.iloc[-1]
                        
                        call_vol = atm_vol_calc + 0.5 * rr_vol + bf_vol
                        put_vol = atm_vol_calc - 0.5 * rr_vol + bf_vol
                        
                        put_pos = delta_order.index(delta)
                        x_positions.append(put_pos)
                        y_values.append(put_vol)
                        hover_labels.append(f'{delta}Δ Put')
                        
                        call_pos = atm_pos + 1 + (len(delta_order) - 1 - delta_order.index(delta))
                        x_positions.append(call_pos)
                        y_values.append(call_vol)
                        hover_labels.append(f'{delta}Δ Call')
            
            if len(x_positions) >= 3:
                sorted_data = sorted(zip(x_positions, y_values, hover_labels))
                x_sorted, y_sorted, labels_sorted = zip(*sorted_data)
                
                fig.add_trace(go.Scatter(
                    x=x_sorted,
                    y=y_sorted,
                    mode='lines+markers',
                    name=f'{tenor} Smile',
                    line=dict(color=colors[i], width=3),
                    marker=dict(size=8),
                    text=labels_sorted,
                    hovertemplate=f'<b>{tenor} %{{text}}</b><br>Vol: %{{y:.1f}}%<extra></extra>'
                ))
        
        if len(fig.data) == 0:
            return self.create_empty_chart("No sufficient data for volatility smiles")
        
        fig.update_layout(
            title=f'USDZAR Volatility Smiles by Tenor - {target_date.strftime("%Y-%m-%d")}',
            xaxis=dict(
                title='Strike Position',
                tickmode='array',
                tickvals=strike_positions,
                ticktext=strike_labels,
                tickangle=45
            ),
            yaxis=dict(title='Implied Volatility (%)'),
            width=1200, height=600,
            plot_bgcolor='white',
            hovermode='closest'
        )
        
        fig.add_vline(x=atm_pos, line_dash="dash", line_color="gray", opacity=0.5)
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        return fig

    def create_smile_term_structure(self):
        """Create term structure for all deltas"""
        target_date = self.get_historical_date(self.lookback_period.value)
        if target_date is None:
            return self.create_empty_chart("No data available")
        
        fig = go.Figure()
        
        tenors = list(fx_tenors.keys())
        colors = ['blue', 'red', 'green', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'black']
        delta_options = ['35DP', '25DP', '15DP', '10DP', 'ATM', '10DC', '15DC', '25DC', '35DC']
        delta_labels = ['35Δ Put', '25Δ Put', '15Δ Put', '10Δ Put', 'ATM', '10Δ Call', '15Δ Call', '25Δ Call', '35Δ Call']
        
        for i, delta in enumerate(delta_options):
            vols = []
            days = []
            for tenor in tenors:
                vol_val = np.nan
                if delta == 'ATM':
                    col_name = f'USDZAR_{tenor}_ATM_IV'
                    if col_name in self.df.columns:
                        data = self.df[col_name].loc[:target_date].dropna()
                        if len(data) > 0:
                            vol_val = data.iloc[-1]
                else:
                    delta_num = delta[:2]
                    option_type = delta[2:]
                    atm_col = f'USDZAR_{tenor}_ATM_IV'
                    rr_col = f'USDZAR_{tenor}_{delta_num}D_RR'
                    bf_col = f'USDZAR_{tenor}_{delta_num}D_BF'
                    if all(col in self.df.columns for col in [atm_col, rr_col, bf_col]):
                        atm_data = self.df[atm_col].loc[:target_date].dropna()
                        rr_data = self.df[rr_col].loc[:target_date].dropna()
                        bf_data = self.df[bf_col].loc[:target_date].dropna()
                        if len(atm_data) > 0 and len(rr_data) > 0 and len(bf_data) > 0:
                            atm_vol = atm_data.iloc[-1]
                            rr_vol = rr_data.iloc[-1]
                            bf_vol = bf_data.iloc[-1]
                            vol_val = atm_vol + (0.5 if option_type == 'DC' else -0.5) * rr_vol + bf_vol
                
                if not pd.isna(vol_val):
                    vols.append(vol_val)
                    days.append(fx_tenors[tenor]['days'])
            
            if vols:
                fig.add_trace(go.Scatter(
                    x=days,
                    y=vols,
                    mode='lines+markers',
                    name=delta_labels[i],
                    line=dict(color=colors[i % len(colors)], width=2),
                    marker=dict(size=6),
                    hovertemplate=f'<b>{delta_labels[i]}</b><br>IV: %{{y:.1f}}%<extra></extra>'
                ))
        
        fig.update_layout(
            title=f'USDZAR Smile Implied Term Structure - {target_date.strftime("%Y-%m-%d")}',
            xaxis=dict(
                title='Tenor (Business Days)',
                type='log',
                tickmode='array',
                tickvals=[1, 5, 10, 15, 21, 42, 63, 84, 126, 189, 252, 378, 504],
                ticktext=['ON', '1W', '2W', '3W', '1M', '2M', '3M', '4M', '6M', '9M', '1Y', '18M', '2Y']
            ),
            yaxis=dict(title='Implied Volatility (%)'),
            width=1000, height=600,
            hovermode='closest',
            plot_bgcolor='white',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        return fig

    def show_time_series_for_tenor(self, tenor):
        """Show comprehensive time series for a specific tenor"""
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=(f"{tenor} ATM Implied vs Realized Volatility", f"{tenor} IV-RV Difference"),
            vertical_spacing=0.15,
            shared_xaxes=True
        )
        
        iv_col = f'USDZAR_{tenor}_ATM_IV'
        rv_col = f'USDZAR_{tenor}_ATM_RV'
        
        if iv_col in self.df.columns:
            iv_series = self.df[iv_col].dropna()
            if len(iv_series) > 0:
                fig.add_trace(go.Scatter(
                    x=iv_series.index, y=iv_series.values,
                    mode='lines', name=f'{tenor} IV',
                    line=dict(color='blue', width=2),
                    hovertemplate=f'{tenor} IV<br>%{{y:.1f}}%<extra></extra>'
                ), row=1, col=1)
        
        if rv_col in self.df.columns:
            rv_series = self.df[rv_col].dropna()
            if len(rv_series) > 0:
                fig.add_trace(go.Scatter(
                    x=rv_series.index, y=rv_series.values,
                    mode='lines', name=f'{tenor} RV',
                    line=dict(color='red', dash='dash', width=2),
                    hovertemplate=f'{tenor} RV<br>%{{y:.1f}}%<extra></extra>'
                ), row=1, col=1)
        
        if iv_col in self.df.columns and rv_col in self.df.columns:
            iv_series = self.df[iv_col].dropna()
            rv_series = self.df[rv_col].dropna()
            
            common_dates = iv_series.index.intersection(rv_series.index)
            if len(common_dates) > 0:
                iv_aligned = iv_series.loc[common_dates]
                rv_aligned = rv_series.loc[common_dates]
                diff_series = iv_aligned - rv_aligned
                
                fig.add_trace(go.Scatter(
                    x=diff_series.index, y=diff_series.values,
                    mode='lines', name=f'{tenor} IV-RV',
                    line=dict(color='orange', width=2),
                    hovertemplate=f'{tenor} IV-RV<br>%{{y:.1f}}vol pts<extra></extra>',
                    showlegend=False
                ), row=2, col=1)
        
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)
        
        fig.update_layout(
            title=f"USDZAR {tenor} Volatility Analysis",
            height=600,
            width=1200,
            plot_bgcolor='white',
            hovermode='x unified'
        )
        
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Volatility (%)", row=1, col=1)
        fig.update_yaxes(title_text="IV-RV (vol pts)", row=2, col=1)
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        # Convert to FigureWidget and place in container
        fig_widget = go.FigureWidget(fig)
        self.time_series_fig_box.children = [fig_widget]
    
    def show_rr_bf_time_series(self, tenor, delta):
        """Show Risk Reversal and Butterfly time series"""
        fig = go.Figure()
        
        rr_col = f'USDZAR_{tenor}_{delta}D_RR'
        bf_col = f'USDZAR_{tenor}_{delta}D_BF'
        
        if rr_col in self.df.columns:
            rr_series = self.df[rr_col].dropna()
            if len(rr_series) > 0:
                fig.add_trace(go.Scatter(
                    x=rr_series.index, y=rr_series.values,
                    mode='lines', name=f'{tenor} {delta}Δ RR',
                    line=dict(color='green', width=2),
                    hovertemplate=f'{tenor} {delta}Δ RR<br>%{{y:.1f}}vol pts<extra></extra>'
                ))
        
        if bf_col in self.df.columns:
            bf_series = self.df[bf_col].dropna()
            if len(bf_series) > 0:
                fig.add_trace(go.Scatter(
                    x=bf_series.index, y=bf_series.values,
                    mode='lines', name=f'{tenor} {delta}Δ BF',
                    line=dict(color='purple', width=2),
                    hovertemplate=f'{tenor} {delta}Δ BF<br>%{{y:.1f}}vol pts<extra></extra>'
                ))
        
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
        
        fig.update_layout(
            title=f'USDZAR {tenor} {delta}Δ Risk Reversal & Butterfly Time Series',
            xaxis=dict(title='Date'),
            yaxis=dict(title='Volatility Points'),
            width=1200, height=400,
            plot_bgcolor='white',
            hovermode='x unified'
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        # Convert to FigureWidget and place in container
        fig_widget = go.FigureWidget(fig)
        self.time_series_fig_box.children = [fig_widget]
   
    def show_smile_time_series(self, tenor, delta_level):
        """Show time series for Call and Put of the same delta level"""
        if delta_level == '50':
            self.show_time_series_for_tenor(tenor)
            return
        
        fig = go.Figure()
        
        atm_col = f'USDZAR_{tenor}_ATM_IV'
        rr_col = f'USDZAR_{tenor}_{delta_level}D_RR'
        bf_col = f'USDZAR_{tenor}_{delta_level}D_BF'
        
        if all(col in self.df.columns for col in [atm_col, rr_col, bf_col]):
            atm_data = self.df[atm_col].dropna()
            rr_data = self.df[rr_col].dropna()
            bf_data = self.df[bf_col].dropna()
            
            common_dates = atm_data.index.intersection(rr_data.index).intersection(bf_data.index)
            
            if len(common_dates) > 0:
                atm_aligned = atm_data.loc[common_dates]
                rr_aligned = rr_data.loc[common_dates]
                bf_aligned = bf_data.loc[common_dates]
                
                call_vol = atm_aligned + 0.5 * rr_aligned + bf_aligned
                put_vol = atm_aligned - 0.5 * rr_aligned + bf_aligned
                
                fig.add_trace(go.Scatter(
                    x=call_vol.index, 
                    y=call_vol.values,
                    mode='lines', 
                    name=f'{tenor} {delta_level}Δ Call IV',
                    line=dict(color='blue', width=2)
                ))
                
                fig.add_trace(go.Scatter(
                    x=put_vol.index, 
                    y=put_vol.values,
                    mode='lines', 
                    name=f'{tenor} {delta_level}Δ Put IV',
                    line=dict(color='red', width=2)
                ))
                
                fig.add_trace(go.Scatter(
                    x=atm_aligned.index, 
                    y=atm_aligned.values,
                    mode='lines', 
                    name=f'{tenor} ATM IV (Reference)',
                    line=dict(color='gray', width=2, dash='dash')
                ))
        
        fig.update_layout(
            title=f'USDZAR {tenor} {delta_level}Δ Call & Put Implied Volatility Time Series',
            xaxis=dict(title='Date'),
            yaxis=dict(title='Implied Volatility (%)'),
            width=1200, height=400,
            plot_bgcolor='white',
            hovermode='x unified',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        # Convert to FigureWidget and place in container
        fig_widget = go.FigureWidget(fig)
        self.time_series_fig_box.children = [fig_widget]
    
    def show_all_tenors_time_series(self):
        """Show time series for ALL tenors"""
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=("ATM Implied vs Realized Volatility (All Tenors)", "IV-RV Difference (All Tenors)"),
            vertical_spacing=0.15,
            shared_xaxes=True
        )
        
        key_tenors = ['1W', '1M', '3M', '6M', '1Y']
        colors = ['blue', 'red', 'green', 'purple', 'orange']
        
        for i, tenor in enumerate(key_tenors):
            iv_col = f'USDZAR_{tenor}_ATM_IV'
            rv_col = f'USDZAR_{tenor}_ATM_RV'
            
            if iv_col in self.df.columns:
                iv_series = self.df[iv_col].dropna()
                if len(iv_series) > 0:
                    fig.add_trace(go.Scatter(
                        x=iv_series.index, y=iv_series.values,
                        mode='lines', name=f'{tenor} IV',
                        line=dict(color=colors[i], width=2),
                        hovertemplate=f'{tenor} IV<br>%{{y:.1f}}%<extra></extra>'
                    ), row=1, col=1)
            
            if rv_col in self.df.columns:
                rv_series = self.df[rv_col].dropna()
                if len(rv_series) > 0:
                    fig.add_trace(go.Scatter(
                        x=rv_series.index, y=rv_series.values,
                        mode='lines', name=f'{tenor} RV',
                        line=dict(color=colors[i], dash='dash', width=2),
                        hovertemplate=f'{tenor} RV<br>%{{y:.1f}}%<extra></extra>'
                    ), row=1, col=1)
            
            if iv_col in self.df.columns and rv_col in self.df.columns:
                iv_series = self.df[iv_col].dropna()
                rv_series = self.df[rv_col].dropna()
                
                common_dates = iv_series.index.intersection(rv_series.index)
                if len(common_dates) > 0:
                    iv_aligned = iv_series.loc[common_dates]
                    rv_aligned = rv_series.loc[common_dates]
                    diff_series = iv_aligned - rv_aligned
                    
                    fig.add_trace(go.Scatter(
                        x=diff_series.index, y=diff_series.values,
                        mode='lines', name=f'{tenor} IV-RV',
                        line=dict(color=colors[i], width=2),
                        hovertemplate=f'{tenor} IV-RV<br>%{{y:.1f}}vol pts<extra></extra>',
                        showlegend=False
                    ), row=2, col=1)
        
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)
        
        fig.update_layout(
            title="USDZAR Volatility Analysis - All Key Tenors",
            height=700,
            width=1200,
            plot_bgcolor='white',
            hovermode='x unified'
        )
        
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Volatility (%)", row=1, col=1)
        fig.update_yaxes(title_text="IV-RV (vol pts)", row=2, col=1)
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
        
        fig_widget = go.FigureWidget(fig)
        self.time_series_fig_box.children = [fig_widget]
    
    def on_heatmap_click(self, trace, points, selector):
        """Handle heatmap cell clicks to display time series"""
        if points.points:
            point = points.points[0]
            chart_type = self.chart_type.value
            tenors = list(fx_tenors.keys())
            
            if chart_type == 'vol_heatmap':
                deltas = ['ATM']
                tenor = tenors[point.x] if point.x < len(tenors) else '1M'
                self.show_time_series_for_tenor(tenor)
            elif chart_type == 'rr_heatmap' or chart_type == 'bf_heatmap':
                deltas = ['10D', '15D', '25D', '35D']
                delta = deltas[point.y] if point.y < len(deltas) else '25D'
                delta_num = delta.replace('D', '')
                tenor = tenors[point.x] if point.x < len(tenors) else '1M'
                if delta_num != '50':
                    self.show_rr_bf_time_series(tenor, delta_num)
            elif chart_type == 'vol_surface_heatmap':
                strike_order = ['35DP', '25DP', '15DP', '10DP', 'ATM', '10DC', '15DC', '25DC', '35DC']
                strike = strike_order[point.x] if point.x < len(strike_order) else 'ATM'
                tenor = tenors[point.y] if point.y < len(tenors) else '1M'
                self.show_time_series_for_tenor(tenor)
            elif chart_type == 'smile_term_structure':
                tenor = tenors[point.x] if point.x < len(tenors) else '1M'
                delta_options = ['35DP', '25DP', '15DP', '10DP', 'ATM', '10DC', '15DC', '25DC', '35DC']
                delta_labels = ['35Δ Put', '25Δ Put', '15Δ Put', '10Δ Put', 'ATM', '10Δ Call', '15Δ Call', '25Δ Call', '35Δ Call']
                selected_delta_idx = point.y if point.y < len(delta_options) else 0
                selected_delta = delta_options[selected_delta_idx]
                delta = selected_delta[:2] if selected_delta != 'ATM' else '50'
                col_name = f'USDZAR_{tenor}_{delta}D_IV' if selected_delta != 'ATM' else f'USDZAR_{tenor}_ATM_IV'
                if col_name in self.df.columns:
                    iv_series = self.df[col_name].dropna()
                    if len(iv_series) > 0:
                        fig = go.Figure()
                        fig.add_trace(go.Scatter(
                            x=iv_series.index, y=iv_series.values,
                            mode='lines', name=f'{tenor} {delta_labels[selected_delta_idx]} IV',
                            line=dict(color='blue', width=2),
                            hovertemplate=f'{tenor} {delta_labels[selected_delta_idx]} IV<br>%{{y:.1f}}%<extra></extra>'
                        ))
                        fig.update_layout(
                            title=f'USDZAR {tenor} {delta_labels[selected_delta_idx]} Implied Volatility Time Series',
                            xaxis=dict(title='Date'),
                            yaxis=dict(title='Implied Volatility (%)'),
                            width=1200, height=400,
                            plot_bgcolor='white',
                            hovermode='x unified'
                        )
                        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
                        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
                        fig_widget = go.FigureWidget(fig)
                        self.time_series_fig_box.children = [fig_widget]
    
    def create_empty_chart(self, message="No data available"):
        """Create an empty chart with a message"""
        fig = go.Figure()
        fig.add_annotation(
            text=message,
            xref="paper", yref="paper",
            x=0.5, y=0.5, showarrow=False,
            font=dict(size=16)
        )
        fig.update_layout(width=1000, height=600)
        return fig
    
    def update_chart(self):
        """Update chart based on selections"""
        if self.df.empty:
            self.main_fig_box.children = [
                widgets.HTML('<p style="color: red;">❌ DataFrame is empty. Please load data first.</p>')
            ]
            return
        
        chart_type = self.chart_type.value
        
        try:
            if chart_type == 'vol_surface_heatmap':
                fig = self.create_vol_surface_heatmap(self.lookback_period.value)
            elif chart_type == 'vol_heatmap':
                fig = self.create_volatility_heatmap(self.lookback_period.value)
            elif chart_type == 'rr_heatmap':
                fig = self.create_rr_heatmap(self.lookback_period.value)
            elif chart_type == 'bf_heatmap':
                fig = self.create_bf_heatmap(self.lookback_period.value)
            elif chart_type == 'atm_term_structure':
                fig = self.create_atm_term_structure()
            elif chart_type == 'rr_term_structure':
                fig = self.create_rr_term_structure()
            elif chart_type == 'bf_term_structure':
                fig = self.create_bf_term_structure()
            elif chart_type == 'vol_smile':
                fig = self.create_vol_smile_by_tenor()
            elif chart_type == 'smile_term_structure':
                fig = self.create_smile_term_structure()
            else:
                fig = self.create_empty_chart(f"Chart type '{chart_type}' not implemented")
            
            # Convert to FigureWidget and place in container
            fig_widget = go.FigureWidget(fig)
            
            # Register click handler for heatmap chart types
            if chart_type in ['vol_surface_heatmap', 'vol_heatmap', 'rr_heatmap', 'bf_heatmap', 'smile_term_structure']:
                if len(fig_widget.data) > 0:
                    fig_widget.data[0].on_click(self.on_heatmap_click)
            
            self.main_fig_box.children = [fig_widget]
            
        except Exception as e:
            self.main_fig_box.children = [
                widgets.HTML(f'<p style="color: red;">Error creating chart: {str(e)}</p>')
            ]
    
    def display(self):
        """Display the dashboard"""
        display(self.dashboard)


def create_fx_vol_surface_dashboard(df):
    """Create FX volatility surface dashboard"""
    dashboard = FXVolSurfaceDashboard(df)
    dashboard.display()
    return None


# =============================================================================
# EXECUTION
# =============================================================================

# Process data
df_fx_surface = process_fx_vol_surface_data()

# Create and display dashboard
create_fx_vol_surface_dashboard(df_fx_surface)

🚀 Starting FX Volatility Surface data processing...
   Processing 108 tickers
   Executing BQL request...
   Data loaded: 3452 dates, 108 columns
✅ Processing complete: (3452, 121)
